In [1]:
import os
import pandas as pd
import vnstock
import warnings
from dotenv import load_dotenv
from time import perf_counter

# Setup

In [2]:
load_dotenv()
vnstock_id = os.getenv("VNSTOCK_ID")
if not vnstock_id:
    raise ValueError("VNSTOCK_ID is not set. Add it to .env")

vnstock.login(vnstock_id);

In [3]:
account_info = vnstock.status()
print(f"Account authenticated status: {account_info['authenticated']}")
print(f"Account tier status: {account_info['tier']}")

Account authenticated status: True
Account tier status: free


## 1.1 Listing

In [4]:
ticker_table = vnstock.Reference().equity.list_by_exchange()

In [5]:
#number of rows and number of unique symbols
print(f"number of rows: {len(ticker_table)}")
print(f"number of unique symbols: {ticker_table['symbol'].nunique()}")

number of rows: 1755
number of unique symbols: 1755


- `symbol` is unique, no NULL symbols

In [6]:
# table information
ticker_table.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1755 entries, 0 to 1754
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   symbol            1755 non-null   string
 1   exchange          1755 non-null   string
 2   organ_name        1728 non-null   string
 3   organ_short_name  1728 non-null   string
 4   icb_code_lv2      1736 non-null   string
dtypes: string(5)
memory usage: 213.9 KB


In [7]:
ticker_table[ticker_table["organ_name"].isna()]

,symbol,exchange,organ_name,organ_short_name,icb_code_lv2
131,VHI,DELISTED,<NA>,<NA>,3500
265,TS5,DELISTED,<NA>,<NA>,<NA>
392,T12,DELISTED,<NA>,<NA>,<NA>
401,SVL,DELISTED,<NA>,<NA>,2700
444,SON,DELISTED,<NA>,<NA>,2700
724,PDT,DELISTED,<NA>,<NA>,7500
788,NNQ,DELISTED,<NA>,<NA>,3500
834,MXC,DELISTED,<NA>,<NA>,<NA>
1020,IPH,DELISTED,<NA>,<NA>,<NA>
1056,HUX,DELISTED,<NA>,<NA>,<NA>


In [8]:
ticker_table[ticker_table["icb_code_lv2"].isna()]

,symbol,exchange,organ_name,organ_short_name,icb_code_lv2
265,TS5,DELISTED,<NA>,<NA>,<NA>
387,TAC,DELISTED,Công ty Cổ phần Dầu Thực vật Tường An,Dầu Tường An,<NA>
392,T12,DELISTED,<NA>,<NA>,<NA>
471,SIC,DELISTED,Công ty Cổ phần ANI,Công ty Cổ phần ANI,<NA>
598,PYU,DELISTED,Công ty Cổ phần Môi trường và Công trình đô th...,Công trình đô thị Phúc Yên,<NA>
834,MXC,DELISTED,<NA>,<NA>,<NA>
1020,IPH,DELISTED,<NA>,<NA>,<NA>
1056,HUX,DELISTED,<NA>,<NA>,<NA>
1151,HGR,DELISTED,<NA>,<NA>,<NA>
1153,HGC,DELISTED,<NA>,<NA>,<NA>


In [9]:
# total ticker per exchange
ticker_table.groupby("exchange").agg({"symbol": "count"})

,symbol
exchange,
DELISTED,239
HNX,299
HOSE,408
UPCOM,809


In [10]:
ticker_table[ticker_table["symbol"].str.len() != 3]

,symbol,exchange,organ_name,organ_short_name,icb_code_lv2


- `organ_name` and `organ_short_name` columns are string columns with both have 27 NULL values.
- `icb_code_lv2` column is a string columns that has 19 NULL values.
- There are 3 different exchanges: HNX, HOSE, UPCOM.
- HNX has 299 active tickers, HOSE has 408 tickers, and UPCOM has 810 tickers.
- DELISTED refer to stocks that were previously listed on one of the three exchanges but are no longer traded.
- All NULL values of `organ_name` and `organ_short_name` are from DELISTED ticker.
- All NULL values of `icb_code_lv2` are from DELISTED ticker.
- All symbols have exactly 3 characters.

## 1.2 OHLCV

Default source for `ohlcv()` is VCI (same as the listing). The pipeline will pass `source` explicitly.

In [11]:
mkt = vnstock.Market()
fpt_ohlcv = mkt.equity("FPT").ohlcv(start="2013-05-22")

c:\Users\Long\Desktop\flagships\.venv\Lib\site-packages\vnstock\ui\market.py:280: UserWarning: This result is limited to what the Community tier includes: the latest 2,920 days of history only. 1,368 rows were left out. Fuller coverage is part of the usage-rights plans: https://vnstocks.com/pricing
  bars = self._call(


In [12]:
fpt_ohlcv

,time,open,high,low,close,volume
0,2018-11-08 07:00:00+07:00,11.96,12.01,11.88,11.94,352130
1,2018-11-09 07:00:00+07:00,11.84,11.92,11.67,11.68,635120
2,2018-11-12 07:00:00+07:00,11.67,11.88,11.65,11.88,273110
3,2018-11-13 07:00:00+07:00,11.67,11.82,11.67,11.79,438030
4,2018-11-14 07:00:00+07:00,11.86,11.96,11.78,11.96,752440
...,...,...,...,...,...,...
1990,2026-11-02 07:00:00+07:00,52.40,52.50,50.10,50.20,12293400
1991,2026-11-03 07:00:00+07:00,50.30,51.70,49.55,50.40,9249300
1992,2026-11-04 07:00:00+07:00,50.60,53.90,50.40,53.90,12822500
1993,2026-11-05 07:00:00+07:00,54.50,54.60,52.80,52.90,8006200


- ohlcv data goes back 2,920 days = 1,995 trading rows (community tier limit)

In [13]:
fpt_ohlcv.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 1995 entries, 0 to 1994
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype                           
---  ------  --------------  -----                           
 0   time    1995 non-null   datetime64[ns, Asia/Ho_Chi_Minh]
 1   open    1995 non-null   float64                         
 2   high    1995 non-null   float64                         
 3   low     1995 non-null   float64                         
 4   close   1995 non-null   float64                         
 5   volume  1995 non-null   int64                           
dtypes: datetime64[ns, Asia/Ho_Chi_Minh](1), float64(4), int64(1)
memory usage: 93.6 KB


- currency unit is thousands of VND/Vietnam Dong (₫)
- the data in ohlcv is "adjusted Data", the old data has odd decimal but new data has clean decimal because company undergoes a split or issues a dividend. 
- 0.047 kb or 47 b a row for one ticker. BigQuery also needs a `symbol` column, so rows will be bigger.

In [14]:
fpt_ohlcv["time"].dt.time.nunique()

1

- The clock part is always 07:00 (+7:00) = 00:00 UTC -> the column means a trading date -> DATE in BigQuery.

In [15]:
fpt_ohlcv["time"].nunique()

1995

- fpt_ohlcv table has 1995 rows, column `time` has 1995 unique values -> `time` is unique.

In [16]:
# Tet 2026 period
fpt_ohlcv[(fpt_ohlcv["time"] >= "2026-02-08") & (fpt_ohlcv["time"] <= "2026-02-27")]

,time,open,high,low,close,volume
1811,2026-02-09 07:00:00+07:00,88.60,89.14,87.79,88.78,5123453
1812,2026-02-10 07:00:00+07:00,89.58,89.58,87.25,87.70,6143120
1813,2026-02-11 07:00:00+07:00,88.06,89.23,87.34,88.42,5525584
1814,2026-02-12 07:00:00+07:00,88.78,88.96,88.24,88.60,4137993
1815,2026-02-13 07:00:00+07:00,87.97,88.33,86.36,86.36,18546292
1816,2026-02-23 07:00:00+07:00,86.80,86.80,83.49,84.74,22701438
1817,2026-02-24 07:00:00+07:00,84.29,84.29,82.95,82.95,28627330
1818,2026-02-25 07:00:00+07:00,82.68,82.95,79.81,79.90,47840750
1819,2026-02-26 07:00:00+07:00,80.71,82.68,80.62,81.15,17593368


- Holidays in Vietnam result in missing rows for the days on which they fall.

In [17]:
fpt_ohlcv[fpt_ohlcv["volume"] == 0]

,time,open,high,low,close,volume


- No `volume` = 0. Check again with smaller ticker in 1.3

open questions:
- adjusted prices + rolling window (3.2)
- 1 machine -> self-hosted runner (5.1)

## 1.3 Intraday trades

Outputs reflect data fetched on 2026-11-06 (trades) / 2026-11-09 (listing, OHLCV). Re-running API cells shows newer data.

In [18]:
# liquid ticker
fpt_intraday_vci = mkt.equity("FPT").trades()

C:\Users\Long\AppData\Local\Temp\ipykernel_24556\1356400360.py:2: UserWarning: Trades of FPT from VCI do not cover the whole session: 1,000 rows, only 14:18:31-14:45:00, missing the start (limit=1,000). VCI answers at most 1,000 trades a call: fetch the page before with last_id=<earliest id>, or use source="kbs", limit=100,000.
  fpt_intraday_vci = mkt.equity("FPT").trades()


- VCI raises error when limit above 1000

In [19]:
fpt_intraday_vci

,time,price,volume,match_type,id
0,2026-11-06 14:18:31+07:00,50.9,2400,Sell,534576792
1,2026-11-06 14:18:31+07:00,50.9,3400,Sell,534576791
2,2026-11-06 14:18:31+07:00,50.9,300,Sell,534576790
3,2026-11-06 14:18:31+07:00,50.9,100,Sell,534576789
4,2026-11-06 14:18:31+07:00,50.9,200,Sell,534576788
...,...,...,...,...,...
995,2026-11-06 14:29:57+07:00,50.9,100,Sell,534624250
996,2026-11-06 14:29:57+07:00,50.9,200,Sell,534624249
997,2026-11-06 14:29:57+07:00,50.9,500,Sell,534624248
998,2026-11-06 14:29:57+07:00,50.9,1400,Sell,534624247


- `vci` returns at most 1,000 rows per call -> tried `kbs` to see if it gives a full day in one call

In [20]:
fpt_intraday_kbs = mkt.equity("FPT").trades(limit=10000, source="kbs")
fpt_intraday_kbs

c:\Users\Long\Desktop\flagships\.venv\Lib\site-packages\vnstock\ui\market.py:329: UserWarning: This result is limited to what the Community tier includes: the newest 5,000 rows only. 1,107 rows were left out. Fuller coverage is part of the usage-rights plans: https://vnstocks.com/pricing
  return self._call("trades", limit=limit, last_id=last_id, side=side, source=source)


,time,price,volume,match_type,id
0,2026-11-06 09:40:23+07:00,51.5,2000,buy,20261106_3316900
1,2026-11-06 09:40:26+07:00,51.4,100,sell,20261106_3317000
2,2026-11-06 09:40:31+07:00,51.5,100,buy,20261106_3317100
3,2026-11-06 09:40:32+07:00,51.4,1000,sell,20261106_3318100
4,2026-11-06 09:40:32+07:00,51.5,1100,buy,20261106_3319200
...,...,...,...,...,...
4995,2026-11-06 14:29:51+07:00,51.0,300,buy,20261106_11378500
4996,2026-11-06 14:29:51+07:00,50.9,200,sell,20261106_11378700
4997,2026-11-06 14:29:54+07:00,50.9,100,sell,20261106_11378800
4998,2026-11-06 14:29:58+07:00,50.9,20000,sell,20261106_11398800


In [47]:
if os.path.exists(f"../data/fpt_intraday_kbs_{fpt_intraday_kbs['time'][0].strftime('%Y-%m-%d')}.parquet"):
    print("file already exists.")
else:
    fpt_intraday_kbs.to_parquet(path=f"../data/fpt_intraday_kbs_{fpt_intraday_kbs.iloc[0, 0].strftime('%Y-%m-%d')}.parquet", index=False)

file already exists.


In [22]:
fpt_intraday_kbs_last_id = mkt.equity("FPT").trades(limit=1000, last_id="20261106_3316900", source="kbs")
fpt_intraday_kbs_last_id

C:\Users\Long\AppData\Local\Temp\ipykernel_24556\2975700802.py:1: UserWarning: Trades of FPT from KBS do not cover the whole session: 1,000 rows, only 14:04:40-14:45:09, missing the start (limit=1,000). Raise limit to 100,000 for more; if the ends still fall short of 09:00/09:15 and 14:45, raise it again.
  fpt_intraday_kbs_last_id = mkt.equity("FPT").trades(limit=1000, last_id="20261106_3316900", source="kbs")


,time,price,volume,match_type,id
0,2026-11-06 14:04:40+07:00,51.2,500,buy,20261106_9649100
1,2026-11-06 14:04:41+07:00,51.1,1000,sell,20261106_9650100
2,2026-11-06 14:04:42+07:00,51.2,5100,buy,20261106_9655200
3,2026-11-06 14:04:44+07:00,51.1,300,sell,20261106_9655500
4,2026-11-06 14:04:45+07:00,51.1,1000,buy,20261106_9656500
...,...,...,...,...,...
995,2026-11-06 14:29:51+07:00,51.0,300,buy,20261106_11378500
996,2026-11-06 14:29:51+07:00,50.9,200,sell,20261106_11378700
997,2026-11-06 14:29:54+07:00,50.9,100,sell,20261106_11378800
998,2026-11-06 14:29:58+07:00,50.9,20000,sell,20261106_11398800


- `kbs`: max 5,000 rows per call on the free tier, and `last_id` is ignored -> can not get a full day for busy tickers
- Decision: `vci` with paging (last_id). `kbs` rejected

In [48]:
# Get missing rows from page 2
fpt_intraday_vci_page2 = mkt.equity("FPT").trades(limit=1000, last_id="534576792", source="vci")

C:\Users\Long\AppData\Local\Temp\ipykernel_24556\3532478900.py:2: UserWarning: Trades of FPT from VCI do not cover the whole session: 1,000 rows, only 14:10:47-14:18:31, missing the start (limit=1,000). VCI answers at most 1,000 trades a call: fetch the page before with last_id=<earliest id>, or use source="kbs", limit=100,000.
  fpt_intraday_vci_page2 = mkt.equity("FPT").trades(limit=1000, last_id="534576792", source="vci")


In [49]:
fpt_intraday_vci_page2

,time,price,volume,match_type,id
0,2026-11-06 14:10:47+07:00,51.0,400,Sell,534548343
1,2026-11-06 14:10:48+07:00,51.0,300,Buy,534548394
2,2026-11-06 14:10:48+07:00,51.0,100,Sell,534548378
3,2026-11-06 14:10:48+07:00,51.0,600,Sell,534548377
4,2026-11-06 14:10:52+07:00,51.0,100,Buy,534548568
...,...,...,...,...,...
995,2026-11-06 14:18:31+07:00,50.9,300,Sell,534576787
996,2026-11-06 14:18:31+07:00,50.9,100,Sell,534576786
997,2026-11-06 14:18:31+07:00,50.9,300,Sell,534576781
998,2026-11-06 14:18:31+07:00,50.9,100,Sell,534576780


In [25]:
fpt_intraday_vci.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype                           
---  ------      --------------  -----                           
 0   time        1000 non-null   datetime64[ns, Asia/Ho_Chi_Minh]
 1   price       1000 non-null   float64                         
 2   volume      1000 non-null   int64                           
 3   match_type  1000 non-null   string                          
 4   id          1000 non-null   string                          
dtypes: datetime64[ns, Asia/Ho_Chi_Minh](1), float64(1), int64(1), string(2)
memory usage: 51.6 KB


In [26]:
#Loop to get whole day worth of rows
start = perf_counter()
safety_net = 0

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    
    fpt_intraday = mkt.equity("FPT").trades(limit=1000, source="vci")
    fpt_intraday["id"]= fpt_intraday["id"].astype("int")

    while True:
        fpt_missing_rows = mkt.equity("FPT").trades(limit=1000, last_id=str(fpt_intraday["id"].min()), source="vci")    
        fpt_missing_rows["id"]= fpt_missing_rows["id"].astype("int")

        if len(fpt_missing_rows) == 0:
            break
        
        fpt_intraday = pd.concat([fpt_missing_rows, fpt_intraday], axis=0, ignore_index=True)

        safety_net += 1
        if safety_net == 50:
            print("safety net break")
            break

print(fpt_intraday)
end = perf_counter()
print(f"total run time: {end - start}")
print(f"safety net: {safety_net}")

                           time  price  volume match_type         id
0     2026-11-06 09:15:00+07:00   51.4     600       Sell  534080834
1     2026-11-06 09:15:00+07:00   51.5     500        Buy  534080827
2     2026-11-06 09:15:00+07:00   51.5     500        Buy  534080823
3     2026-11-06 09:15:00+07:00   51.4  370500        ATO  534080562
4     2026-11-06 09:15:01+07:00   51.5     200        Buy  534080931
...                         ...    ...     ...        ...        ...
14609 2026-11-06 14:29:57+07:00   50.9     100       Sell  534624250
14610 2026-11-06 14:29:57+07:00   50.9     200       Sell  534624249
14611 2026-11-06 14:29:57+07:00   50.9     500       Sell  534624248
14612 2026-11-06 14:29:57+07:00   50.9    1400       Sell  534624247
14613 2026-11-06 14:45:00+07:00   50.8  763400        ATC  534625524

[14614 rows x 5 columns]
total run time: 172.9432314000005
safety net: 14


In [46]:
# save dataframe as parquet file to data folder
if os.path.exists(f"../data/fpt_intraday_{fpt_intraday['time'][0].strftime('%Y-%m-%d')}.parquet"):
    print("file already exists.")
else:
    fpt_intraday.to_parquet(path=f"../data/fpt_intraday_{fpt_intraday.iloc[0, 0].strftime('%Y-%m-%d')}.parquet", index=False)

file already exists.


In [28]:
fpt = pd.read_parquet("../data/fpt_intraday_2026-11-06.parquet")
fpt

,time,price,volume,match_type,id
0,2026-11-06 09:15:00+07:00,51.4,600,Sell,534080834
1,2026-11-06 09:15:00+07:00,51.5,500,Buy,534080827
2,2026-11-06 09:15:00+07:00,51.5,500,Buy,534080823
3,2026-11-06 09:15:00+07:00,51.4,370500,ATO,534080562
4,2026-11-06 09:15:01+07:00,51.5,200,Buy,534080931
...,...,...,...,...,...
14609,2026-11-06 14:29:57+07:00,50.9,100,Sell,534624250
14610,2026-11-06 14:29:57+07:00,50.9,200,Sell,534624249
14611,2026-11-06 14:29:57+07:00,50.9,500,Sell,534624248
14612,2026-11-06 14:29:57+07:00,50.9,1400,Sell,534624247


In [29]:
# check if ids overlap
fpt["id"].nunique()

14614

- `id` is unique

In [30]:
# total values by match type
fpt.groupby("match_type").agg("count")

,time,price,volume,id
match_type,,,,
ATC,1,1,1,1
ATO,1,1,1,1
Buy,5567,5567,5567,5567
Sell,9045,9045,9045,9045


In [31]:
#compare ohlcv and intraday
fpt_ohlcv[fpt_ohlcv["time"].dt.normalize() == "2026-11-06"]

,time,open,high,low,close,volume
1994,2026-11-06 07:00:00+07:00,51.4,51.8,50.8,50.8,12162200


In [32]:
fpt.agg({"volume":"sum"})

volume    12162100
dtype: int64

- `ohlcv` volume is 100 more than intraday volume

In [33]:
fpt[fpt["match_type"] == "ATC"]

,time,price,volume,match_type,id
14613,2026-11-06 14:45:00+07:00,50.8,763400,ATC,534625524


`ATC` price = `ohlcv close`

In [34]:
fpt[fpt["match_type"] == "ATO"]

,time,price,volume,match_type,id
3,2026-11-06 09:15:00+07:00,51.4,370500,ATO,534080562


`ATO` price = `ohlcv open`

- The missing volume exploration: `notebooks/trades_reconciliation.ipynb`
- `vci` recorded one trade as 500 instead of 600 at `14:22:56`, not a paging bug

- ATO (opening auction) and ATC (closing auction)
- There is no trading at 14:30-14:45 because that is ATC collection window. Orders come in but nothing matches until 14:45, when it all happens in one big match.

In [35]:
fpt.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 14614 entries, 0 to 14613
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype                           
---  ------      --------------  -----                           
 0   time        14614 non-null  datetime64[ns, Asia/Ho_Chi_Minh]
 1   price       14614 non-null  float64                         
 2   volume      14614 non-null  int64                           
 3   match_type  14614 non-null  string                          
 4   id          14614 non-null  int64                           
dtypes: datetime64[ns, Asia/Ho_Chi_Minh](1), float64(1), int64(2), string(1)
memory usage: 622.6 KB


- 622.6 KB for 14614 lines -> 43.6 B per rows